[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DSP/ICA_Blind_Source_Separation.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Blind Source Separation & ICA

The cocktail-party problem, actually solved: several microphones each hear a *mixture* of sources, and — knowing nothing about the mixing — we unmix them. The key is a beautiful statistical loophole: Gaussianity is the one thing mixing *increases*. Verified the only way that matters: recovered sources correlate ≈1 with the planted truth.

## 1. Pre-requisites

[Statistical SP](./Statistical_Signal_Processing.ipynb), [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S3, [Independence](../Intro_Math/Analysis/Independence.ipynb).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

# three planted sources: a chirp 'voice', a square-wave 'hum', an impulsive 'percussion'
fs, T_dur = 8000, 3.0
t = np.arange(0, T_dur, 1/fs)
s1 = sig.chirp(t, 300, T_dur, 800) * (1 + 0.3*np.sin(2*np.pi*2*t))
s2 = sig.square(2*np.pi*120*t) * 0.7
s3 = np.zeros_like(t)
for tc in rng.uniform(0, T_dur, 25):
    i = int(tc*fs); s3[i:i+150] += np.exp(-np.arange(150)/25) * rng.choice([-2, 2])
S_true = np.stack([s1, s2, s3])
S_true = (S_true - S_true.mean(1, keepdims=True)) / S_true.std(1, keepdims=True)

A_mix = rng.standard_normal((3, 3))                    # unknown room acoustics
X = A_mix @ S_true                                      # what the microphones record

---
### 🕐 Session 1 of 3 — *The Problem & Why Correlation Isn't Enough* (~35 min)
**Goal:** see mixing destroy the sources; understand why PCA/whitening only gets you halfway.
**Feeds into:** Session 2 (the non-Gaussian loophole).

---

## 2. Three Microphones, Three Tangles

💡 **Intuition.** Each mic hears $x_i = \sum_j a_{ij} s_j$: linear, instantaneous mixing. **Whitening** (decorrelating via the covariance [eigendecomposition](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb)) can undo mixing *up to a rotation* — but second-order statistics are **rotation-blind**: every rotation of white signals is equally white. Correlation has taken you to a sphere of candidate unmixings and gone silent. Something beyond variance must pick the rotation — that something is Session 2.

In [ ]:
# whiten

# YOUR CODE HERE


**What just happened.** The top row shows three structured sources — a chirp, a square wave, sparse impulsive bursts — and the bottom row shows what the microphones actually record: three tangles in which none of that structure is visible. Mixing destroyed the signals as far as the eye is concerned.

Then whitening runs, and the printed check confirms it worked in its own terms: the covariance of $Z$ is the identity to within 1e-10. All second-order structure has been removed. But the next line is the important one — the correlation of the first whitened channel with the three true sources is **0.72, 0.63, 0.28**. Channel 1 is still a blend of at least two sources. Whitening decorrelated the data without unmixing it.

**Why it necessarily stops there.** If $Z$ has covariance $I$, then so does $QZ$ for *any* orthogonal matrix $Q$, because $Q I Q^\top = I$. Whitening therefore determines the solution only **up to a rotation** — it has narrowed infinitely many candidate unmixings down to a sphere of them, and every point on that sphere is exactly as white as every other. Second-order statistics are *rotation-blind*, and no amount of cleverness with covariance will break the tie. The 0.72/0.63 pair is that unresolved rotation, measured.

This is worth stating as a general limit rather than a quirk of this dataset. PCA and whitening exhaust everything covariance knows; if the structure you need survives in the residual rotation, you must look beyond second order to find it.

**Which is also why the problem looked ill-posed to begin with.** For any invertible $R$, $X = (AR)(R^{-1}S)$ is a perfectly good factorisation, so nothing in $X$ alone singles out the true $A$ and $S$. Some additional assumption must do that work. Session 2 supplies it — and remarkably, the assumption that the sources are *statistically independent and non-Gaussian* turns out to be sufficient.

One design note to keep in mind: the three sources here were chosen to be strongly non-Gaussian in different ways — a chirp, a square wave, and sparse bursts. That is not incidental. It is exactly the resource Session 2 will spend, and Session 3 shows what happens when it is unavailable.

---
### 🕐 Session 2 of 3 — *The Non-Gaussian Loophole & FastICA* (~40 min)
**Goal:** the CLT in reverse: mixtures are MORE Gaussian than sources — so maximize non-Gaussianity.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (limits & practice).

---

## 3. Gaussianity as a Compass

💡 **Intuition.** The [CLT](../Intro_Math/Analysis/Independence.ipynb) says sums of independent things drift *toward* Gaussian. Flip it around: each microphone (a sum of sources) is **more Gaussian than any single source** — so to unmix, rotate the whitened data until each output is as **non-Gaussian as possible**. That's all of ICA. FastICA does it with a fixed-point iteration on a smooth non-Gaussianity score (we use $\log\cosh$), one source at a time, deflating (Gram–Schmidt) so each new direction is orthogonal to the found ones. Built-in limits fall out of the logic: source order and sign/scale are unrecoverable, and **two Gaussian sources cannot be separated** (their mixtures are exactly rotation-symmetric).

In [ ]:
# ORACLE: each recovered source must match ONE true source with |corr| ≈ 1

# YOUR CODE HERE


**What just happened.** Per-source best correlations of **[1.000, 1.000, 0.9993]** against the planted truth — the cocktail party, solved. And the correlation *matrix* is the real evidence, not those three numbers:

```
[[1.    0.015 0.   ]
 [0.003 0.035 1.   ]
 [0.008 0.999 0.008]]
```

Each row has exactly one near-1 entry and near-zeros elsewhere, and the argmax positions form a **permutation** — recovered source 1 ↔ true source 1, 2 ↔ 3, 3 ↔ 2. That is a much stronger claim than "the correlations are high." A degenerate solution in which two outputs both latched onto the loudest source would still show some large correlations; it would fail the permutation check, which is exactly why the `assert` tests `len(set(corr.argmax(1))) == 3`. Off-diagonal entries at the 0.01 level mean each output contains essentially none of the other sources.

**What the algorithm was not given.** `A_mix` is never referenced after `X` is constructed. No knowledge of the mixing, the room, the number of microphones relative to sources, the sources' spectra, or their timing entered the computation. The only assumption was that the sources are statistically independent and non-Gaussian. That is what makes this *blind* separation, and it is worth pausing on: an ill-posed factorisation became uniquely solvable on the strength of one statistical assumption.

**Why the assumption is enough.** The [CLT](../Intro_Math/Analysis/Independence.ipynb) says sums of independent quantities drift toward Gaussian, so each microphone — being a sum of sources — is *more Gaussian than any source in it*. Gaussianity is monotone under mixing: there is no way to become less Gaussian by mixing more. So "rotate until each output is maximally non-Gaussian" is a compass that can only point toward unmixing. FastICA follows it with a fixed-point iteration on $\log\cosh$ (hence the `tanh` in the code), deflating by Gram–Schmidt so each new direction is orthogonal to those already found — legitimate because after whitening, independent directions *are* orthogonal.

**Two things ICA structurally cannot recover, visible in the output above.** The **order** is permuted rather than preserved, and the **sign and scale** are arbitrary — which is why the comparison uses `np.abs` and why the plot below computes a `flip`. Neither permuting sources nor negating one changes their independence or their non-Gaussianity, so nothing in the problem statement distinguishes those solutions. These are genuine properties of blind separation, not defects of the implementation, and any downstream use has to tolerate them.

**And a caveat on how favourable this is.** The three sources were chosen to be strongly non-Gaussian in different ways, the mixing is instantaneous (a plain matrix, with no delays), and there are exactly as many microphones as sources. Session 3 removes the first of those conditions and watches the method fail.

In [ ]:

# YOUR CODE HERE


**What just happened.** The recovered traces sit on top of the dashed truth almost everywhere — the chirp's sweep, the square wave's hard edges, and the sparse bursts all reconstructed from three microphone signals that individually looked like noise.

The waveform view adds something the correlation numbers cannot: it shows *where* the reconstruction is good. Correlation is a single summary, and a value of 0.999 could in principle hide a badly-reconstructed transient. Here the square wave's discontinuities and the percussion's sharp onsets are reproduced cleanly, which matters because those are precisely the features a separation method that had merely captured the dominant subspace would smear.

**Note the two corrections the plot has to make.** `order = corr.argmax(1)` re-sorts the outputs to match the truth, and `flip = np.sign(...)` fixes each trace's sign. Both are necessary because ICA cannot recover permutation or sign — negating a source or reordering the set changes neither its independence nor its non-Gaussianity, so nothing distinguishes those solutions. Being explicit about this is the honest way to plot: the alignment is applied for *display*, and no information from the truth was used during separation.

That distinction is worth insisting on, because it is a place where a demo can quietly cheat. The unmixing matrix came from `fastica(Z, 3)`, which sees only the whitened microphone data. The true sources appear afterwards, twice: once to compute the correlation oracle, and once to order and flip the traces for this figure. Neither influenced the result.

**What this looks like in the field.** EEG artifact removal runs exactly this pipeline — eye blinks are gloriously non-Gaussian, so ICA isolates them into their own component, which is then simply deleted before reconstructing the clean signal. Same three lines, same assumptions.

Session 3 now removes the assumption that made all of this work.

---
### 🕐 Session 3 of 3 — *Limits, Diagnostics & Practice* (~30 min)
**Goal:** what ICA can't do, how to sanity-check it, and where it runs in the wild.
**Builds on:** Session 2.

---

## 4. The Honest Fine Print

In [ ]:
# the promised failure: two GAUSSIAN sources are unseparable — watch it happen

# YOUR CODE HERE


**Field guide.**

- **Works:** EEG artifact removal (eye blinks are gloriously non-Gaussian), [audio](./Audio_Speech_DSP.ipynb) unmixing with instantaneous mixtures, hyperspectral unmixing.
- **Fails or needs upgrades:** convolutive/reverberant mixing (rooms delay, not just scale — needs frequency-domain ICA), more sources than mics (underdetermined → [sparsity](./Sparse_Dictionary_Learning.ipynb) to the rescue), Gaussian-ish sources.
- **Diagnostics:** always check kurtosis of outputs (should be far from 0), and run restarts — consistent answers across restarts are the practical identifiability certificate.
- **Lineage:** [contrastive learning](../Intro_Mach_Learn/Representation_Learning.ipynb) and modern disentanglement research are ICA's descendants (nonlinear ICA is provably impossible without auxiliary structure — a live research frontier).

## 5. Conclusion

Whitening gets you to a rotation; the CLT-in-reverse picks it; FastICA computes it (recovered × truth correlations > 0.99, verified); and Gaussian sources mark the hard boundary of the possible (also verified). Blindness, it turns out, is negotiable — Gaussianity isn't.

---
## Where next

- [Array Processing](./Array_Processing.ipynb) — unmixing with *geometry* instead of statistics.
- [Representation Learning](../Intro_Mach_Learn/Representation_Learning.ipynb) — the neural descendants.
- [Manifold Optimization](../Intro_Math/Optimization/Manifold_Optimization.ipynb) — ICA's rotation search lives on the Stiefel manifold.